# Dataset review

What the model receives: samples from `MammogramDataset` as they come out of the
data loaders, with and without augmentation. Needs the manifest, `splits.json` and
the preprocessed data (see `docs/data-pipeline.md`). Outputs are stripped before
commit (`nbstripout`).

The questions: does the augmentation keep the mask on the lesion and look
plausible, does the padding sit next to the breast, and does the sampler give the
intended share of images with a mass?


In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from mammo_seg.data.augmentation import Augmentation, AugmentationSettings
from mammo_seg.data.dataset import MammogramDataset
from mammo_seg.data.folds import fold_ids
from mammo_seg.data.loaders import evaluation_loader, training_loader
from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.paths import INBREAST_PROCESSED_DIR
from mammo_seg.data.splits import read_splits


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
PROCESSED = ROOT / INBREAST_PROCESSED_DIR
manifest = read_manifest(PROCESSED / "manifest.json")
splits_file = read_splits(PROCESSED / "splits.json")
FOLD = 0
ids = fold_ids(manifest, splits_file, FOLD)
positives = {image.file_id for image in manifest.images if image.has_mass}
print(f"fold {FOLD}: {len(ids.train)} training, {len(ids.validation)} validation")

## One image, several augmentations

The first panel is the sample without augmentation; the others are random
augmentations of the same image. Red is the mask outline, blue shades the padding
(`valid` is false there).


In [ ]:
SETTINGS = AugmentationSettings()  # the defaults; change them to compare
FILE_ID = next(file_id for file_id in ids.train if file_id in positives)

plain = MammogramDataset(PROCESSED / "preprocessed", [FILE_ID])
augmented = MammogramDataset(
    PROCESSED / "preprocessed", [FILE_ID], transform=Augmentation(SETTINGS)
)


def show(ax, sample, title: str) -> None:
    ax.imshow(sample["image"][0], cmap="gray", vmin=0, vmax=1)
    padding = np.ma.masked_where(
        sample["valid"][0].numpy(), np.ones(sample["valid"].shape[1:])
    )
    ax.imshow(padding, cmap="winter", alpha=0.3, vmin=0, vmax=1)
    if sample["mask"].any():
        ax.contour(sample["mask"][0], levels=[0.5], colors="red", linewidths=0.8)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


torch.manual_seed(0)
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
show(axes[0], plain[0], f"{FILE_ID} without augmentation")
for k, ax in enumerate(axes[1:]):
    show(ax, augmented[0], f"augmentation {k + 1}")
plt.tight_layout()
plt.show()

## A training batch

A batch as the training loader draws it: augmented and balanced by the sampler.


In [ ]:
train_dataset = MammogramDataset(
    PROCESSED / "preprocessed", ids.train, transform=Augmentation(SETTINGS)
)
loader = training_loader(train_dataset, manifest, batch_size=8, seed=0)
batch = next(iter(loader))

fig, axes = plt.subplots(1, 8, figsize=(22, 5))
for k, ax in enumerate(axes):
    sample = {key: batch[key][k] for key in ("image", "mask", "valid")}
    has = "mass" if batch["file_id"][k] in positives else "no mass"
    show(ax, sample, f"{batch['file_id'][k]} ({has})")
plt.tight_layout()
plt.show()

## What the sampler draws over an epoch

The share of images with a mass in one training epoch with the sampler, against
the natural share in the training and validation folds.


In [ ]:
counts = Counter()
for batch in training_loader(train_dataset, manifest, batch_size=8, seed=0):
    counts.update(file_id in positives for file_id in batch["file_id"])
drawn = counts[True] + counts[False]
natural = len(positives & set(ids.train)) / len(ids.train)
validation = evaluation_loader(
    MammogramDataset(PROCESSED / "preprocessed", ids.validation), batch_size=8
)
seen = [file_id for batch in validation for file_id in batch["file_id"]]
print(
    f"training epoch: {drawn} draws, {counts[True] / drawn:.2f} with a mass "
    f"(natural share {natural:.2f})"
)
print(
    f"validation: {len(seen)} images, {len(positives & set(seen)) / len(seen):.2f} "
    "with a mass, every image once"
)